## Run ID for experiment

In [1]:
# --- Pipeline Run ID ---
from datetime import datetime, timezone
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

## Installation

In [2]:
!apt-get install -y gdal-bin libgdal-dev 2>/dev/null | tail -3
!pip install -q rasterio==1.3.10 shapely psycopg2-binary boto3 numpy tqdm
!pip install boto3 botocore
!pip uninstall -y dataproc-spark-connect pyspark
!pip install pyspark==3.5.1 apache-sedona
!pip install -q imageio
import site
import importlib
importlib.reload(site)


Setting up python3-gdal (3.8.4+dfsg-1~jammy0) ...
Setting up gdal-bin (3.8.4+dfsg-1~jammy0) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires rasterio>=1.4, but you have rasterio 1.3.10 which is incompatible.
Found existing installation: dataproc-spark-connect 1.0.2
Uninstalling dataproc-spark-connect-1.0.2:
  Successfully uninstalled dataproc-spark-connect-1.0.2
Found existing installation: pyspark 4.

<module 'site' (frozen)>

## Params

In [3]:
from google.colab import userdata
import os
EVAL_START = '2026-01'
EVAL_END   = '2026-03'
MIN_BASELINE_MONTHS = 12
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME')
AWS_REGION            = 'us-west-1'
S3_COG_FOLDERS = ['50m_resolution/cog/']
SCENE_FETCH_LOGIC = 'concurrent'
DB_CONFIG = {
    'host':     userdata.get('DB_HOST'),
    'port':     5432,
    'database': 'postgres',
    'user':     'postgres',
    'password': userdata.get('DB_PASSWORD'),
    'connect_timeout': 10
}
Z_SCORE_THRESHOLD = -1.5
PIXEL_STRIDE = 10
FOREST_MASK_PATH = '/tmp/worldcover_boulder.tif'
# all_bounds = (-180.0, -90.0, 180.0, 90.0) # for a spatiotemporal query of cog's
# all_start_date = datetime.date(1900, 1, 1)
# all_end_date   = datetime.date.today() + datetime.timedelta(days=365)
LOCAL_WORK_DIR = '/tmp/vegetation_pipeline'
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
DRIVE_SOURCE_DIR = '/content/drive/MyDrive/BigDataProject'
print('AWS credentials loaded.')

import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 pyspark-shell"

AWS credentials loaded.


In [4]:
import csv, os
params_dict = {
    'EVAL_START': EVAL_START,
    'EVAL_END': EVAL_END,
    'MIN_BASELINE_MONTHS': MIN_BASELINE_MONTHS,
    'S3_BUCKET_NAME': S3_BUCKET_NAME,
    'AWS_REGION': AWS_REGION,
    'S3_COG_FOLDERS': str(S3_COG_FOLDERS),
    'SCENE_FETCH_LOGIC': SCENE_FETCH_LOGIC,
    'Z_SCORE_THRESHOLD': Z_SCORE_THRESHOLD,
    'PIXEL_STRIDE': PIXEL_STRIDE,
    'FOREST_MASK_PATH': FOREST_MASK_PATH
}

params_dir = f"results/run_id={run_id}/metrics/{run_id}"
os.makedirs(params_dir, exist_ok=True)
params_csv_path = os.path.join(params_dir, "params.csv")

with open(params_csv_path, mode="w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Parameter", "Value"])
    for k, v in params_dict.items():
        writer.writerow([k, v])

print(f"Saved {len(params_dict)} parameters to {params_csv_path}")
import boto3
s3_client = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
s3_key = f'results/run_id={run_id}/params.csv'
s3_client.upload_file(params_csv_path, S3_BUCKET_NAME, s3_key)
print(f'Uploaded params to s3://{S3_BUCKET_NAME}/{s3_key}')


Saved 10 parameters to results/run_id=20260317_055040/metrics/20260317_055040/params.csv
Uploaded params to s3://vegetation-anomaly-cogs/results/run_id=20260317_055040/params.csv


## Metric Logging

In [5]:
import time
import csv
import os
import json as _json
from urllib.request import urlopen

_prev_spark_snapshot = {}

def _spark_executor_stats(spark):
    """Query the Spark REST API for live executor metrics."""
    try:
        ui_url = spark.sparkContext.uiWebUrl
        app_id = spark.sparkContext.applicationId
        url = f"{ui_url}/api/v1/applications/{app_id}/executors"
        resp = urlopen(url, timeout=5)
        return _json.loads(resp.read().decode('utf-8'))
    except Exception:
        return []

def log_metrics(stage, start_time=None, extra_metrics=None):
    """
    Collect and log evaluation metrics for each pipeline stage.

    Scalability      : elapsed_time_sec, processing_throughput_mb_per_sec
    Concurrency      : spark_parallelism, num_executors,
                        total_executor_task_time_ms, concurrency_ratio
    Query Efficiency : query_latency_sec (= elapsed_time_sec)
    """
    global run_id, _prev_spark_snapshot
    metrics = {"run_id": run_id, "stage": stage,
               "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}

    elapsed = None
    if start_time is not None:
        elapsed = time.time() - start_time
        metrics["elapsed_time_sec"] = round(elapsed, 2)
        metrics["query_latency_sec"] = round(elapsed, 2)

    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        metrics["spark_parallelism"] = spark.sparkContext.defaultParallelism
        # metrics["num_executors"] is populated below using the REST API response

        executors = _spark_executor_stats(spark)
        if executors:
            metrics["num_executors"] = len(executors)
        if executors:
            total_task_ms = sum(e.get('totalDuration', 0) for e in executors)
            prev_task_ms = _prev_spark_snapshot.get('total_task_ms', 0)
            delta_task_ms = total_task_ms - prev_task_ms
            metrics["total_executor_task_time_ms"] = delta_task_ms

            if elapsed and elapsed > 0:
                metrics["concurrency_ratio"] = round(delta_task_ms / (elapsed * 1000), 2)

            total_input_bytes = sum(e.get('totalInputBytes', 0) for e in executors)
            prev_bytes = _prev_spark_snapshot.get('total_input_bytes', 0)
            delta_mb = (total_input_bytes - prev_bytes) / (1024 * 1024)
            if elapsed and elapsed > 0:
                metrics["processing_throughput_mb_per_sec"] = round(delta_mb / elapsed, 2)

            _prev_spark_snapshot['total_input_bytes'] = total_input_bytes
            _prev_spark_snapshot['total_task_ms'] = total_task_ms
    except Exception as ex:
        metrics["spark_metrics_error"] = str(ex)

    if extra_metrics:
        metrics.update(extra_metrics)

    print(f"\n=== {stage} Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print("====================\n")

    csv_dir = f"results/run_id={run_id}/metrics/{run_id}"
    csv_path = os.path.join(csv_dir, "pipeline_metrics.csv")
    os.makedirs(csv_dir, exist_ok=True)
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=sorted(metrics.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(metrics)

    import boto3
    try:
        if 'AWS_ACCESS_KEY_ID' in globals():
            s3_client = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
            s3_key = f'results/run_id={run_id}/pipeline_metrics.csv'
            s3_client.upload_file(csv_path, S3_BUCKET_NAME, s3_key)
            print(f'  [S3] Metrics uploaded to s3://{S3_BUCKET_NAME}/{s3_key}')
    except Exception as e:
        print(f'  [S3] Failed to upload metrics: {e}')
    return metrics


## Initialise Spark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg

spark = (
    SparkSession.builder
    .appName("VegetationPipeline")
    .config("spark.driver.memory","4g")
    .config("spark.hadoop.fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.endpoint","s3.amazonaws.com")
    .config("spark.hadoop.fs.s3a.path.style.access","true")
    .getOrCreate()
)
print(f"Spark initialized: {spark.version}")


Spark initialized: 3.5.1


## Worldcover data

In [7]:
import boto3
s3 = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
s3.download_file(S3_BUCKET_NAME, 'worldcover/worldcover_boulder.tif', '/tmp/worldcover_boulder.tif')
print('WorldCover tile downloaded to /tmp/worldcover_boulder.tif')

WorldCover tile downloaded to /tmp/worldcover_boulder.tif


## Auto-discovery (Unused)

In [8]:
#     # ── PostGIS discovery ──
# --- PostGIS discovery (commented out — using S3_COG_FOLDERS instead) ---
# def get_db_connection(config):
#     db_config = config['db_config']
#     try:
#         conn = psycopg2.connect(
#             host=db_config['host'],
#             port=db_config['port'],
#             database=db_config['database'],
#             user=db_config['user'],
#             password=db_config['password'],
#             connect_timeout=db_config.get('connect_timeout', 10)
#         )
#         conn.autocommit = False

#         cur = conn.cursor()
#         cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
#         cur.execute("SELECT PostGIS_Version();")
#         postgis_ver = cur.fetchone()[0]
#         cur.execute("SELECT version();")
#         pg_ver = cur.fetchone()[0].split(' ')[1]

#         print(f"  Database connected successfully!")
#         print(f"  PostgreSQL version: {pg_ver}")
#         print(f"  PostGIS version: {postgis_ver.split(' ')[0]}")
#         cur.close()
#         return conn

#     except psycopg2.OperationalError as e:
#         print(f"  [ERROR] Cannot connect to database: {e}")
#         print(f"  Check: 1) RDS endpoint correct? 2) Security group allows port 5432?")
#         print(f"         3) RDS instance is 'Available' in AWS console?")
#         return None

# def run_spatiotemporal_query(conn, west, south, east, north,
#                               start_date, end_date):
#     cur = conn.cursor()
#     cur.execute("""
#         SELECT
#             tile_id,
#             acquisition_date,
#             file_url,
#             ndvi_mean,
#             resolution_m,
#             ST_AsText(geom) AS bbox_wkt
#         FROM vegetation_metadata
#         WHERE
#             ST_Intersects(
#                 geom,
#                 ST_MakeEnvelope(%s, %s, %s, %s, 4326)
#             )
#             AND acquisition_date BETWEEN %s AND %s
#         ORDER BY acquisition_date;
#     """, (west, south, east, north, start_date, end_date))

#     results = cur.fetchall()
#     cur.close()
#     return results
#     conn_transform = get_db_connection(config=CONFIG)
#     if conn_transform:
#         scene_fetch_start = time.time()
#         print("Fetching all COG URLs from PostGIS database...")
#         results = run_spatiotemporal_query(
#             conn_transform,
#             *all_bounds,
#             all_start_date, all_end_date
#         )
#         conn_transform.close()
#         print(f'Found {len(results)} scene(s) in the database.')

#         log_metrics("Scene Fetch", scene_fetch_start, {"Scenes fetched": len(results), "scenes_per_sec": round(len(results) / (time.time() - scene_fetch_start), 2)})
#     else:
#         print('Failed to connect to database. Cannot fetch scene URLs.')
#         results = []

## Collecting & Pre-processing Scenes

In [9]:
import numpy as np
import boto3
import rasterio
from rasterio.session import AWSSession
from rasterio.transform import Affine
import re

def extract_scene_date(tile_id, file_url):
    """
    Extract scene date from file_url/tile_id patterns.
    Supported patterns:
      1) veg_YYYY-MM-DD
      2) HLS.<prod>.<tile>.YYYYDOY...
      3) tile_id suffix _YYYYDDD
    Returns datetime.date or None.
    """
    url = str(file_url)
    tid = str(tile_id)

    m = re.search(r'veg_([0-9]{4})-([0-9]{2})-([0-9]{2})', url)
    if m:
        y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return datetime.date(y, mo, d)

    m = re.search(r'HLS\.[SL]30\.[A-Z0-9]+\.([0-9]{4})([0-9]{3})T', url)
    if m:
        y, doy = int(m.group(1)), int(m.group(2))
        return datetime.date(y, 1, 1) + datetime.timedelta(days=doy - 1)

    m = re.search(r'_([0-9]{7})$', tid)
    if m:
        y = int(m.group(1)[:4])
        doy = int(m.group(1)[4:])
        return datetime.date(y, 1, 1) + datetime.timedelta(days=doy - 1)

    return None

def strip_date_from_tile_id(tile_id):
    """
    Convert date-stamped tile ids like veg_2017-10-01 into stable ids like veg.
    Leave non-matching ids unchanged.
    """
    return re.sub(r'_[0-9]{4}-[0-9]{2}-[0-9]{2}$', '', str(tile_id))

from concurrent.futures import ThreadPoolExecutor, as_completed
def read_bands_from_s3(results, aws_key, aws_secret, region='us-west-1',
                        red_band=1, nir_band=2, swir1_band=3, pixel_stride=10):
    """
    Reads Red, NIR, and SWIR1 band arrays directly from S3 COGs for each
    scene in results. Streams via rasterio AWSSession — no local download.

    Returns list of (tile_id, parsed_date, red, nir, swir1, bbox_wkt,
                      strided_transform, crs_str) per scene.
    strided_transform and crs_str are needed by apply_forest_mask.
    """
    def process_scene(scene):
        tile_id, _db_acq_date, file_url, _, _res, bbox_wkt = scene
        parsed_date = extract_scene_date(tile_id, file_url)
        if parsed_date is None:
            print(f'  [WARN] Could not parse date from path/id. Skipping scene: {tile_id} -> {file_url}')
            return None

        boto3_session = boto3.Session(aws_access_key_id=aws_key, aws_secret_access_key=aws_secret, region_name=region)
        aws_session = AWSSession(boto3_session)

        try:
            with rasterio.Env(aws_session):
                with rasterio.open(file_url) as src:
                    n = src.count
                    rb, nb, sb = min(red_band, n), min(nir_band, n), min(swir1_band, n)
                    red   = src.read(rb)[::pixel_stride, ::pixel_stride].astype('float32')
                    nir   = src.read(nb)[::pixel_stride, ::pixel_stride].astype('float32')
                    swir1 = src.read(sb)[::pixel_stride, ::pixel_stride].astype('float32')
                    ds_mask = src.dataset_mask()[::pixel_stride, ::pixel_stride]

                    invalid = ds_mask == 0
                    for arr in (red, nir, swir1):
                        arr[invalid] = np.nan

                    t = src.transform
                    strided_transform = Affine(t.a * pixel_stride, t.b, t.c, t.d, t.e * pixel_stride, t.f)
                    crs_str = src.crs

            print(f'  ✓ Read {tile_id} ({parsed_date})')
            return (tile_id, parsed_date, red, nir, swir1, bbox_wkt, strided_transform, crs_str)
        except Exception as e:
            print(f'  [ERROR] Failed reading {tile_id}: {e}')
            return None

    scene_bands = []
    if SCENE_FETCH_LOGIC == 'serial':
        print(f'Fetching {len(results)} scenes serially...')
        for r in results:
            res = process_scene(r)
            if res is not None:
                scene_bands.append(res)
    else:
        print(f'Fetching {len(results)} scenes concurrently...')
        with ThreadPoolExecutor(max_workers=24) as executor:
            futures = {executor.submit(process_scene, r): r for r in results}
            for future in as_completed(futures):
                res = future.result()
                if res is not None:
                    scene_bands.append(res)

    return scene_bands
import datetime
import psycopg2
from psycopg2 import sql
from google.colab import userdata
import os

CONFIG = {
    'aws_access_key_id':     AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
    'aws_region':            AWS_REGION,
    's3_bucket':             S3_BUCKET_NAME,
    'db_config':             DB_CONFIG,
    'phase':                 1,
    'drive_source_dir':      DRIVE_SOURCE_DIR,
    'local_work_dir':        LOCAL_WORK_DIR,
    'upload_to_s3':          True,
    'skip_existing':         True,
    'cog_compress':          'LZW',
    'cog_blocksize':         512,
    'cog_overviews':         [2, 4, 8, 16],
    'compute_ndvi_preview':  True,
    'ndvi_sample_size':      1000,
}

def list_cogs_from_s3(bucket, folders, aws_key, aws_secret, region='us-west-1'):
    """
    List all COG .tif files from the given S3 folders.
    Returns results in the same 6-tuple format as run_spatiotemporal_query:
      (tile_id, acq_date_placeholder, file_url, None, None, None)
    so read_bands_from_s3() can consume them unchanged.
    """
    s3_client = boto3.client(
        's3', aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret, region_name=region)
    results = []
    for prefix in folders:
        prefix = prefix.rstrip('/') + '/'
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                if not key.lower().endswith('.tif'):
                    continue
                fname = key.rsplit('/', 1)[-1]
                tile_id = re.sub(r'\.cog\.tif$|\.tif$', '', fname, flags=re.IGNORECASE)
                file_url = f's3://{bucket}/{key}'
                results.append((tile_id, None, file_url, None, None, None))
    print(f'Listed {len(results)} COG(s) from S3 folders: {folders}')
    return results

scene_fetch_start = time.time()
results = list_cogs_from_s3(
    S3_BUCKET_NAME, S3_COG_FOLDERS,
    AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION)
log_metrics("Scene Fetch", scene_fetch_start, {"Scenes fetched": len(results), "scenes_per_sec": round(len(results) / (time.time() - scene_fetch_start), 2)})

if results:
    scene_bands = read_bands_from_s3(
        results,
        AWS_ACCESS_KEY_ID,
        AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION,
        red_band=1, nir_band=2, swir1_band=3,
        pixel_stride=PIXEL_STRIDE,
    )
    print(f'Bands loaded from S3 for {len(scene_bands)} scene(s). No download needed.')
else:
    print('No results found or database connection failed. Skipping band reading.')
    scene_bands = []
def build_pixel_dataframe(sedona, scene_bands):
    """
    Flattens per-scene band arrays into a Spark DataFrame with one row per pixel.
    Columns: tile_id, date, red, nir, swir1, pixel_id, true_pixel_id, bbox_wkt, lon, lat
    NaN pixels (incl. non-forest pixels zeroed by apply_forest_mask) are dropped.
    lon/lat are the pixel centroid coordinates derived from the strided_transform (index 6).
    """
    pixel_rows = []
    for scene in scene_bands:
        tile_id, acq_date, red, nir, swir1, bbox_wkt = scene[:6]
        transform = scene[6] if len(scene) > 6 else None
        true_tile_id = strip_date_from_tile_id(tile_id)
        h, w = red.shape
        for r in range(h):
            for c in range(w):
                rv, nv, sv = float(red[r, c]), float(nir[r, c]), float(swir1[r, c])
                if rv != rv or nv != nv or sv != sv:
                    continue
                if transform is not None:
                    lon = float(transform.c + c * transform.a)
                    lat = float(transform.f + r * transform.e)
                else:
                    lon, lat = None, None
                pixel_rows.append((
                    tile_id, acq_date,
                    rv, nv, sv,
                    f'{tile_id}_{r}_{c}',
                    f'{true_tile_id}_{r}_{c}',
                    bbox_wkt, lon, lat,
                ))
    print(f'  Pixel rows (non-NaN): {len(pixel_rows):,}')
    from pyspark.sql.types import StructType, StructField, StringType, FloatType, DoubleType, DateType
    schema = StructType([
        StructField('tile_id', StringType(), True),
        StructField('date', DateType(), True),
        StructField('red', FloatType(), True),
        StructField('nir', FloatType(), True),
        StructField('swir1', FloatType(), True),
        StructField('pixel_id', StringType(), True),
        StructField('true_pixel_id', StringType(), True),
        StructField('bbox_wkt', StringType(), True),
        StructField('lon', DoubleType(), True),
        StructField('lat', DoubleType(), True)
    ])
    return sedona.createDataFrame(pixel_rows, schema=schema)

from rasterio.warp import reproject, Resampling

def apply_forest_mask(scene_bands, worldcover_path, forest_class=10):
    """
    Masks each scene's band arrays to forested pixels only using ESA WorldCover.
    Class 10 = Tree cover. Non-forest pixels are set to NaN.

    Operates on numpy arrays (before Spark) by reprojecting the WorldCover
    GeoTIFF onto each scene's subsampled pixel grid using rasterio.
    No Sedona, no geoparquet, no geom column required.

    Parameters
    ----------
    scene_bands     : list of 8-tuples from read_bands_from_s3()
                      (..., strided_transform, crs_str)
    worldcover_path : local path to the ESA WorldCover GeoTIFF
    forest_class    : WorldCover class for tree cover (default 10)
    """
    masked = []
    with rasterio.open(worldcover_path) as wc:
        for scene in scene_bands:
            tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs = scene
            red, nir, swir1 = red.copy(), nir.copy(), swir1.copy()
            h, w = red.shape
            forest_mask = np.zeros((h, w), dtype=np.uint8)
            reproject(
                source=rasterio.band(wc, 1),
                destination=forest_mask,
                dst_transform=transform,
                dst_crs=crs,
                resampling=Resampling.nearest,
            )
            is_forest = (forest_mask == forest_class)
            for arr in (red, nir, swir1):
                arr[~is_forest] = np.nan
            n_forest = int(is_forest.sum())
            print(f'  [{tile_id}] Forest pixels: {n_forest:,} / {h*w:,} '
                  f'({100*n_forest/max(h*w,1):.1f}%)')
            masked.append((tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs))
    return masked

def compute_vegetation_indices(df):
    """
    Computes NDVI and NDMI from Red, NIR, and SWIR1 bands.
    NDVI = (NIR - Red)  / (NIR + Red)
    NDMI = (NIR - SWIR1) / (NIR + SWIR1)
    """
    from pyspark.sql.functions import col, when
    return (
        df
        .withColumn('ndvi',
            when((col('nir') + col('red')) != 0,
                 (col('nir') - col('red')) / (col('nir') + col('red')))
            .otherwise(None))
        .withColumn('ndmi',
            when((col('nir') + col('swir1')) != 0,
                 (col('nir') - col('swir1')) / (col('nir') + col('swir1')))
            .otherwise(None))
    )

def harmonize_time_series(df):
    """
    Aggregates irregular observations into monthly composites.
    Returns a regular time series for each stable pixel.
    """
    return (
        df.groupBy('true_pixel_id', year('date').alias('year'), month('date').alias('month'))
          .agg(avg('ndvi').alias('ndvi_mean'), avg('ndmi').alias('ndmi_mean'))
          .dropna(subset=['ndvi_mean', 'ndmi_mean'], how='all')
          .orderBy('year', 'month')
    )

if 'scene_bands' in locals() and scene_bands:
    transform_start = time.time()
    forest_data_path = FOREST_MASK_PATH
    masked_bands = apply_forest_mask(scene_bands, forest_data_path)
    spark_df   = build_pixel_dataframe(spark, masked_bands)
    indices_df = compute_vegetation_indices(spark_df)
    final_df   = harmonize_time_series(indices_df)
    final_df.show(10)
    print(f'Transformation Complete: {final_df.count()} monthly records generated.')
    pixel_count = spark_df.count()
    monthly_count = final_df.count()
    log_metrics("Transformation", transform_start, {"Pixel records": pixel_count, "Monthly records": monthly_count})
else:
    print('No scene_bands found. Run the S3 band-read cell above first.')
    final_df = None


Listed 584 COG(s) from S3 folders: ['50m_resolution/cog/']

=== Scene Fetch Metrics ===
  run_id: 20260317_055040
  stage: Scene Fetch
  timestamp: 2026-03-17 05:52:30
  elapsed_time_sec: 0.32
  query_latency_sec: 0.32
  spark_parallelism: 2
  num_executors: 1
  total_executor_task_time_ms: 8032
  concurrency_ratio: 25.4
  processing_throughput_mb_per_sec: 0.0
  Scenes fetched: 584
  scenes_per_sec: 1846.72

  [S3] Metrics uploaded to s3://vegetation-anomaly-cogs/results/run_id=20260317_055040/pipeline_metrics.csv
Fetching 584 scenes concurrently...
  ✓ Read veg_2015-12-02 (2015-12-02)
  ✓ Read veg_2016-07-29 (2016-07-29)
  ✓ Read veg_2016-04-10 (2016-04-10)
  ✓ Read veg_2016-07-19 (2016-07-19)
  ✓ Read veg_2016-07-09 (2016-07-09)
  ✓ Read veg_2016-04-30 (2016-04-30)
  ✓ Read veg_2016-01-01 (2016-01-01)
  ✓ Read veg_2015-08-09 (2015-08-09)
  ✓ Read veg_2016-08-13 (2016-08-13)
  ✓ Read veg_2016-02-05 (2016-02-05)
  ✓ Read veg_2016-02-25 (2016-02-25)
  ✓ Read veg_2016-01-31 (2016-01-31)


## Anomaly Detection

In [10]:
anomaly_start = time.time()
from pyspark.sql.functions import stddev_pop as stddev, mean as spark_mean, when, col
import pyspark.sql.functions as F

if final_df is None:
    raise ValueError('final_df is not defined. Upstream transformation did not produce any rows.')

def parse_ym(ym_str):
    y, m = ym_str.split('-')
    return int(y), int(m)

eval_start_year, eval_start_month = parse_ym(EVAL_START)
start_ym_int = eval_start_year * 100 + eval_start_month

data_min = final_df.agg(F.min(col('year') * 100 + col('month'))).collect()[0][0]
data_start_year, data_start_month = data_min // 100, data_min % 100
baseline_months = (eval_start_year - data_start_year) * 12 + (eval_start_month - data_start_month)

if baseline_months < MIN_BASELINE_MONTHS:
    raise ValueError(
        f'Insufficient baseline: only {baseline_months} month(s) before EVAL_START={EVAL_START}. '
        f'Need at least {MIN_BASELINE_MONTHS}. Move EVAL_START later or ingest more historical scenes.'
    )
print(f'Baseline window: {data_start_year}-{data_start_month:02d} to {EVAL_START} ({baseline_months} months)')

baseline_df = (
    final_df.filter((col('year') * 100 + col('month')) < start_ym_int)
    .groupBy('true_pixel_id', 'month')
    .agg(
        spark_mean('ndvi_mean').alias('ndvi_mu'),
        stddev('ndvi_mean').alias('ndvi_sigma'),
        spark_mean('ndmi_mean').alias('ndmi_mu'),
        stddev('ndmi_mean').alias('ndmi_sigma'),
    )
)

if EVAL_END:
    eval_end_year, eval_end_month = parse_ym(EVAL_END)
    end_ym_int = eval_end_year * 100 + eval_end_month
    eval_filter = (col('year') * 100 + col('month')).between(start_ym_int, end_ym_int)
else:
    eval_filter = (col('year') * 100 + col('month')) >= start_ym_int
eval_df = final_df.filter(eval_filter)
print(f'Evaluation window: {EVAL_START} to {EVAL_END or "end"}  ({eval_df.count()} pixel-months)')

anomaly_df = eval_df.join(baseline_df, on=['true_pixel_id', 'month'], how='left')

anomaly_df = anomaly_df.withColumn(
    'ndvi_zscore',
    when(col('ndvi_sigma') > 0,
         (col('ndvi_mean') - col('ndvi_mu')) / col('ndvi_sigma'))
    .otherwise(0.0)
).withColumn(
    'ndmi_zscore',
    when(col('ndmi_sigma') > 0,
         (col('ndmi_mean') - col('ndmi_mu')) / col('ndmi_sigma'))
    .otherwise(0.0)
)

anomaly_df = anomaly_df.withColumn(
    'is_ndvi_anomaly', col('ndvi_zscore') < Z_SCORE_THRESHOLD
).withColumn(
    'is_ndmi_anomaly', col('ndmi_zscore') < Z_SCORE_THRESHOLD
)

anomalies = anomaly_df.filter(col('is_ndvi_anomaly') | col('is_ndmi_anomaly'))
anomalies.show(20)
print(f'Total anomalies detected: {anomalies.count()}')
print(f'Total records evaluated: {anomaly_df.count()}')

anomaly_count = anomalies.count()
anomaly_eval_count = anomaly_df.count()
log_metrics("Anomaly Detection", anomaly_start, {"Anomalies detected": anomaly_count, "Total records evaluated": anomaly_eval_count})


Baseline window: 2015-12 to 2026-01 (121 months)
Evaluation window: 2026-01 to 2026-03  (7534 pixel-months)
+-------------+-----+----+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------------------+-------------------+---------------+---------------+
|true_pixel_id|month|year|          ndvi_mean|           ndmi_mean|             ndvi_mu|          ndvi_sigma|            ndmi_mu|          ndmi_sigma|       ndvi_zscore|        ndmi_zscore|is_ndvi_anomaly|is_ndmi_anomaly|
+-------------+-----+----+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------------------+-------------------+---------------+---------------+
|     veg_0_28|    1|2026| 0.6547269709350111|  0.6313989313989313| 0.47761408819494794| 0.16276468788424484| 0.7252328000831811| 0.05814050886271898|1.0881529958514249| -1.613915504348436|          false|           true|
|   

{'run_id': '20260317_055040',
 'stage': 'Anomaly Detection',
 'timestamp': '2026-03-17 06:02:06',
 'elapsed_time_sec': 136.06,
 'query_latency_sec': 136.06,
 'spark_parallelism': 2,
 'num_executors': 1,
 'total_executor_task_time_ms': 136283,
 'concurrency_ratio': 1.0,
 'processing_throughput_mb_per_sec': 0.0,
 'Anomalies detected': 2739,
 'Total records evaluated': 7534}

## Timelapse

In [11]:
import imageio, io
import numpy as np
import matplotlib.pyplot as plt

def generate_timelapse(scene_bands, output_path='/tmp/timelapse.gif',
                        mode='ndvi', fps=3, start_ym=None, end_ym=None):
    """
    Creates a GIF timelapse from scene_bands.

    mode: 'ndvi'        — NDVI heatmap  (green=healthy, red=stressed)
          'ndmi'        — NDMI heatmap  (blue=moist, yellow=dry)
          'false_color' — NRG composite (NIR→R, Red→G, SWIR1→B;
                           vegetation=magenta, bare soil=greenish)
    start_ym / end_ym : 'YYYY-MM' strings to filter scenes (inclusive).
    """
    scenes = sorted(scene_bands, key=lambda s: s[1])  # sort by acq_date
    if start_ym:
        sy, sm = (int(x) for x in start_ym.split('-'))
        scenes = [s for s in scenes if (s[1].year, s[1].month) >= (sy, sm)]
    if end_ym:
        ey, em = (int(x) for x in end_ym.split('-'))
        scenes = [s for s in scenes if (s[1].year, s[1].month) <= (ey, em)]
    if not scenes:
        raise ValueError(f'No scenes in timeframe [{start_ym}, {end_ym}].')

    def _norm(arr):
        lo, hi = np.nanpercentile(arr, 2), np.nanpercentile(arr, 98)
        return np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)

    def _index(a, b):
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where((a + b) > 0, (a - b) / (a + b), np.nan)

    frames = []
    for tile_id, acq_date, red, nir, swir1, *_ in scenes:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=80)
        ax.set_title(f'{acq_date}', fontsize=10, fontweight='bold')
        ax.axis('off')
        if mode == 'ndvi':
            img = ax.imshow(_index(nir, red), cmap='RdYlGn', vmin=-0.2, vmax=0.8)
            plt.colorbar(img, ax=ax, fraction=0.03, pad=0.02, label='NDVI')
        elif mode == 'ndmi':
            img = ax.imshow(_index(nir, swir1), cmap='RdBu', vmin=-0.5, vmax=0.5)
            plt.colorbar(img, ax=ax, fraction=0.03, pad=0.02, label='NDMI')
        elif mode == 'false_color':
            rgb = np.nan_to_num(np.dstack([_norm(nir), _norm(red), _norm(swir1)]))
            ax.imshow(rgb)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        frames.append(imageio.v3.imread(buf))

    imageio.mimsave(output_path, frames, fps=fps, loop=0)
    print(f'Timelapse saved: {output_path}  ({len(frames)} frames @ {fps} fps)')
    return output_path

if 'masked_bands' in vars() and masked_bands:
    from datetime import datetime, timezone
    tl_run_id = run_id
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    for tl_mode, tl_name in [('ndvi', 'ndvi'), ('ndmi', 'ndmi'), ('false_color', 'false_color')]:
        local_path = f'/tmp/timelapse_{tl_name}.gif'
        generate_timelapse(
            masked_bands,
            output_path=local_path,
            mode=tl_mode,
            fps=3,
            start_ym=EVAL_START,
            end_ym=EVAL_END,
        )
        s3_key = f'results/run_id={tl_run_id}/timelapse_{tl_name}.gif'
        s3_client.upload_file(local_path, S3_BUCKET_NAME, s3_key)
        print(f'  Uploaded to s3://{S3_BUCKET_NAME}/{s3_key}')
else:
    print('masked_bands not found. Run the data loading cell first.')


Timelapse saved: /tmp/timelapse_ndvi.gif  (4 frames @ 3 fps)
  Uploaded to s3://vegetation-anomaly-cogs/results/run_id=20260317_055040/timelapse_ndvi.gif
Timelapse saved: /tmp/timelapse_ndmi.gif  (4 frames @ 3 fps)
  Uploaded to s3://vegetation-anomaly-cogs/results/run_id=20260317_055040/timelapse_ndmi.gif
Timelapse saved: /tmp/timelapse_false_color.gif  (4 frames @ 3 fps)
  Uploaded to s3://vegetation-anomaly-cogs/results/run_id=20260317_055040/timelapse_false_color.gif


## Anomaly Metadata export

In [12]:
from datetime import datetime, timezone
from pyspark.sql.functions import lit, stddev_pop, mean as spark_mean, when, col

if 'anomaly_df' not in vars() and 'anomaly_pop_df' in vars():
    anomaly_df = anomaly_pop_df  # fall back to population stddev df

if 'S3_BUCKET_NAME' not in vars() or not S3_BUCKET_NAME:
    raise ValueError('S3_BUCKET_NAME is missing.')


anomaly_events = anomaly_df.filter(
    col('is_ndvi_anomaly') | col('is_ndmi_anomaly')
).select(
    lit(run_id).alias('run_id'),
    col('true_pixel_id'),
    col('year').cast('int'),
    col('month').cast('int'),
    col('ndvi_mean'),
    col('ndmi_mean'),
    col('ndvi_mu'),
    col('ndvi_sigma'),
    col('ndmi_mu'),
    col('ndmi_sigma'),
    col('ndvi_zscore'),
    col('ndmi_zscore'),
    col('is_ndvi_anomaly'),
    col('is_ndmi_anomaly'),
)

anomaly_path = f's3a://{S3_BUCKET_NAME}/results/run_id={run_id}/anomaly_events_parquet'
anomaly_events.coalesce(1).write.mode('overwrite').parquet(anomaly_path)
print(f'Anomaly events written to: {anomaly_path}')
print(f'  Rows: {anomaly_events.count()}')


plot_df = anomaly_df.select(
    lit(run_id).alias('run_id'),
    col('true_pixel_id'),
    col('year').cast('int'),
    col('month').cast('int'),
    col('ndvi_mean'),
    col('ndmi_mean'),
    col('ndvi_mu'),
    col('ndvi_sigma'),
    col('ndmi_mu'),
    col('ndmi_sigma'),
    col('ndvi_zscore'),
    col('ndmi_zscore'),
    col('is_ndvi_anomaly'),
    col('is_ndmi_anomaly'),
)

plot_path = f's3a://{S3_BUCKET_NAME}/results/run_id={run_id}/plot_stats_parquet'
plot_df.coalesce(1).write.mode('overwrite').parquet(plot_path)
print(f'Full plot stats written to: {plot_path}')
print(f'  Rows: {plot_df.count()}')

from pyspark.sql.functions import avg, sum as spark_sum, count

monthly_agg = anomaly_df.groupBy('year', 'month').agg(
    avg('ndvi_mean').alias('avg_ndvi'),
    avg('ndmi_mean').alias('avg_ndmi'),
    avg('ndvi_mu').alias('avg_ndvi_baseline'),
    avg('ndmi_mu').alias('avg_ndmi_baseline'),
    avg('ndvi_sigma').alias('avg_ndvi_sigma'),
    avg('ndmi_sigma').alias('avg_ndmi_sigma'),
    spark_sum(col('is_ndvi_anomaly').cast('int')).alias('ndvi_anomaly_count'),
    spark_sum(col('is_ndmi_anomaly').cast('int')).alias('ndmi_anomaly_count'),
    count('*').alias('total_pixels'),
).orderBy('year', 'month')

monthly_path = f's3a://{S3_BUCKET_NAME}/results/run_id={run_id}/monthly_stats_parquet'
monthly_agg.coalesce(1).write.mode('overwrite').parquet(monthly_path)
print(f'Monthly aggregate stats written to: {monthly_path}')
monthly_agg.show(36)

if 'spark_df' not in vars():
    raise ValueError('spark_df is not defined. Re-run the transformation cell first.')

pixel_coords = (
    spark_df
    .select('true_pixel_id', 'lon', 'lat')
    .dropDuplicates(['true_pixel_id'])
    .filter(col('lon').isNotNull() & col('lat').isNotNull())
)
coords_path = f's3a://{S3_BUCKET_NAME}/results/run_id={run_id}/pixel_coords_parquet'
pixel_coords.coalesce(1).write.mode('overwrite').parquet(coords_path)
print(f'Pixel coord lookup written to: {coords_path}')
print(f'  Unique pixels: {pixel_coords.count()}')
pixel_coords.show(5)


Anomaly events written to: s3a://vegetation-anomaly-cogs/results/run_id=20260317_055040/anomaly_events_parquet
  Rows: 2739
Full plot stats written to: s3a://vegetation-anomaly-cogs/results/run_id=20260317_055040/plot_stats_parquet
  Rows: 7534
Monthly aggregate stats written to: s3a://vegetation-anomaly-cogs/results/run_id=20260317_055040/monthly_stats_parquet
+----+-----+------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------+
|year|month|          avg_ndvi|           avg_ndmi| avg_ndvi_baseline| avg_ndmi_baseline|     avg_ndvi_sigma|     avg_ndmi_sigma|ndvi_anomaly_count|ndmi_anomaly_count|total_pixels|
+----+-----+------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------+
|2026|    1|0.4808072727821548|0.41027025477778006|0.3533669007064601|0.5630048581693808|0.10